In [1]:
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234",
    database="company_db"
)

cursor = connection.cursor()

print("Database connected successfully!")

Database connected successfully!


In [2]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS bank_accounts (
    account_id INT PRIMARY KEY,
    customer_name VARCHAR(100),
    balance DECIMAL(10,2)
)
""")

connection.commit()

print("Bank accounts table created successfully!")

Bank accounts table created successfully!


In [3]:
accounts = [
    (101, "Gaurav", 10000),
    (102, "Rahul", 5000)
]

sql = """
INSERT INTO bank_accounts
(account_id, customer_name, balance)
VALUES (%s, %s, %s)
"""

cursor.executemany(sql, accounts)
connection.commit()

print("Bank accounts inserted successfully!")

Bank accounts inserted successfully!


In [4]:
try:
    connection.start_transaction()

    cursor.execute("""
    UPDATE bank_accounts
    SET balance = balance - 2000
    WHERE account_id = 101
    """)

    cursor.execute("""
    UPDATE bank_accounts
    SET balance = balance + 2000
    WHERE account_id = 102
    """)

    connection.commit()

    print("Money transferred successfully!")

except mysql.connector.Error as error:
    connection.rollback()
    print("Transaction failed:", error)

Money transferred successfully!


In [5]:
try:
    sender_id = 101
    receiver_id = 102
    amount = 15000

    connection.start_transaction()

    cursor.execute(
        "SELECT balance FROM bank_accounts WHERE account_id = %s",
        (sender_id,)
    )

    result = cursor.fetchone()

    if result is None:
        raise ValueError("Sender account not found")

    balance = result[0]

    if amount <= 0:
        raise ValueError("Transfer amount must be positive")

    if balance < amount:
        raise ValueError("Insufficient balance")

    cursor.execute("""
    UPDATE bank_accounts
    SET balance = balance - %s
    WHERE account_id = %s
    """, (amount, sender_id))

    cursor.execute("""
    UPDATE bank_accounts
    SET balance = balance + %s
    WHERE account_id = %s
    """, (amount, receiver_id))

    if cursor.rowcount == 0:
        raise ValueError("Receiver account not found")

    connection.commit()
    print("Transfer successful!")

except (mysql.connector.Error, ValueError) as error:
    connection.rollback()
    print("Transfer failed:", error)

Transfer failed: Insufficient balance


In [6]:
cursor.execute("""
SELECT account_id, customer_name, balance
FROM bank_accounts
""")

records = cursor.fetchall()

print("BANK ACCOUNT DETAILS")
print("-" * 45)

for account in records:
    print(
        "ID:", account[0],
        "| Name:", account[1],
        "| Balance:", account[2]
    )

BANK ACCOUNT DETAILS
---------------------------------------------
ID: 101 | Name: Gaurav | Balance: 8000.00
ID: 102 | Name: Rahul | Balance: 7000.00
